## Notebook purpose

In this notebook, I run **inference** with my finetuned **GGUF** model using **llama.cpp** (via `llama-server` / OpenAI-compatible API) to generate **fixed Python code** from `incorrect_code` samples in my **synthetic evaluation dataset** (a local JSON file).

I follow a **Phase 1 (generation-only)** workflow:
- Send each `incorrect_code` to the model (Setting A: no traceback hint).
- Parse the model output and save the predicted fixed code into `eval_outputs/setting_A/<sid>/prediction.py`.
- Save the raw model response, prompt, and metadata for later analysis.

I do **not execute** the predicted scripts in this notebook to avoid long runtimes (many samples involve ML training) and to reduce GPU cost. Execution-based evaluation (runtime success, traceback collection, similarity metrics, etc.) is done later offline in VS Code.

### Tools used
- **llama.cpp**: `llama-server` to serve the GGUF model with GPU offloading.
- **Python + requests**: to call the local OpenAI-compatible endpoint (`/v1/chat/completions`).
- **Local JSON dataset**: reads the synthetic dataset from disk.
- **Filesystem output**: saves per-sample artifacts for reproducible evaluation.


### Imports

In [ ]:
# --- Python standard library: filesystem / OS ---
import os
from pathlib import Path

# --- Python standard library: data / randomness / timing ---
import json
import random
import time
import hashlib
import re

# --- Python standard library: processes ---
import subprocess

# --- Python standard library: structured data ---
from dataclasses import dataclass

# --- Third-party ---
from datasets import Dataset


### Load synthetic dataset and create a reproducible train/eval split

In this cell, I set a fixed random seed, load my **local synthetic JSON dataset**, convert it into a `datasets.Dataset`, and create a deterministic **train/eval split** (15% for evaluation) using the same seed for reproducibility.


In [10]:
SEED = 42
random.seed(SEED)

PROJECT_ROOT = Path.cwd()  
DATA_PATH = (PROJECT_ROOT.parent.parent / "Datasets" / "final_dataset.json").resolve()

with open(DATA_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)


print("Samples:", len(data))
print("Keys:", list(data[0].keys()))

dataset = Dataset.from_list(data)
split = dataset.train_test_split(test_size=0.15, seed=SEED)
train_dataset = split["train"]
eval_dataset  = split["test"]

print("Train:", len(train_dataset))
print("Eval :", len(eval_dataset))

Samples: 582
Keys: ['title', 'description', 'difficulty', 'correct_code', 'incorrect_code', 'error_type']
Train: 494
Eval : 88


### Set up llama.cpp (CUDA build) and configure the local llama-server endpoint

In these cells, I (1) define paths and inference settings for my **GGUF** model, (2) ensure **llama.cpp** exists under `/opt/llama.cpp` and build it with **CUDA** enabled, (3) locate the `llama-server` binary, and (4) create a convenient **symlink** at `./llama.cpp` inside my project workspace so I can reference llama.cpp from the project without consuming workspace volume space.


In [93]:
import os, subprocess, time, requests
from pathlib import Path

# --------- EDIT THESE ----------
MODEL_GGUF = Path("/workspace/MentorApp/outputs/GGUF/qwen2_5_coder_7b_merged_f16.gguf") 
assert MODEL_GGUF.exists(), f"Model not found: {MODEL_GGUF}"

LLAMA_DIR = Path("/opt/llama.cpp")
LLAMA_SERVER_CANDIDATES = [
    LLAMA_DIR / "build/bin/llama-server",
    LLAMA_DIR / "build/bin/server",
]

# llama-server endpoint (INSIDE the GPU machine)
HOST = "127.0.0.1"     # keep local
PORT = 8081
BASE_URL = f"http://{HOST}:{PORT}/v1"

# Inference settings
CTX = 4096             # start here; increase later if prompts truncate
MAX_TOKENS = 768       # IMPORTANT: don't use 8000 for this task
TEMP = 0.0
NGL = 99              # offload as many layers as possible to GPU

print("MODEL_GGUF:", MODEL_GGUF)
print("BASE_URL:", BASE_URL)
print("CTX:", CTX, "MAX_TOKENS:", MAX_TOKENS, "NGL:", NGL)


MODEL_GGUF: /workspace/MentorApp/outputs/GGUF/qwen2_5_coder_7b_merged_f16.gguf
BASE_URL: http://127.0.0.1:8081/v1
CTX: 4096 MAX_TOKENS: 768 NGL: 99


In [34]:
def sh(cmd: str):
    print(">>", cmd)
    subprocess.check_call(cmd, shell=True)

# Clone if needed
if not LLAMA_DIR.exists():
    sh(f"mkdir -p /opt && git clone https://github.com/ggml-org/llama.cpp {LLAMA_DIR}")

# Build with CUDA
sh(f"cd {LLAMA_DIR} && rm -rf build && cmake -B build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON")
sh(f"cd {LLAMA_DIR} && cmake --build build -j")

# Find server binary
SERVER_BIN = next((p for p in LLAMA_SERVER_CANDIDATES if p.exists()), None)
assert SERVER_BIN is not None, f"Couldn't find llama-server. Tried: {LLAMA_SERVER_CANDIDATES}"

print("✅ SERVER_BIN:", SERVER_BIN)


>> cd /opt/llama.cpp && rm -rf build && cmake -B build -DCMAKE_BUILD_TYPE=Release -DGGML_CUDA=ON
-- The C compiler identification is GNU 13.3.0
-- The CXX compiler identification is GNU 13.3.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detecting CXX compile features - done
-- Found Git: /usr/bin/git (found version "2.43.0") 


CMAKE_BUILD_TYPE=Release


-- The ASM compiler identification is GNU
-- Found assembler: /usr/bin/cc
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD
-- Performing Test CMAKE_HAVE_LIBC_PTHREAD - Success
-- Found Threads: TRUE  
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- Found OpenMP_C: -fopenmp (found version "4.5") 
-- Found OpenMP_CXX: -fopenmp (found version "4.5") 
-- Found OpenMP: TRUE (found version "4.5")  
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- Found CUDAToolkit: /usr/local/cuda/targets/x86_64-linux/include (found version "13.0.88") 
-- CUDA Toolkit found
-- The CUDA compiler identification is NVIDIA 13.0.88
-- Detecting CUDA compiler ABI info
-- Detecting CUDA compiler ABI info - done
-- Check for working CUDA compiler: /usr/local/cuda/bin/nvcc - skipped
-- Detecting CUDA compile features
-- Detecting CUDA compi

In [35]:
repo_link = Path("/workspace/MentorApp/llama.cpp")

if repo_link.exists() or repo_link.is_symlink():
    # only remove if it's not already the desired symlink
    if repo_link.is_symlink() and repo_link.resolve() == LLAMA_DIR.resolve():
        print("✅ symlink already correct:", repo_link, "->", repo_link.resolve())
    else:
        print("Removing existing:", repo_link)
        sh(f"rm -rf {repo_link}")

if not repo_link.exists():
    sh(f"ln -s {LLAMA_DIR} {repo_link}")
    print("✅ created symlink:", repo_link, "->", repo_link.resolve())


✅ symlink already correct: /workspace/MentorApp/llama.cpp -> /opt/llama.cpp


### Start llama-server and create a LangChain chat wrapper

In this section, I start `llama-server` **once** (keeping the GGUF model loaded in VRAM for fast inference), verify the server is reachable via `/v1/models`, and print initial startup logs to confirm CUDA/GPU offloading.

Then I initialize a **LangChain** `ChatOpenAI` client pointing to the local OpenAI-compatible endpoint and define a small `server_chat()` helper to send **system + user** messages and return the model’s text output.


In [104]:
import socket

def is_port_open(host, port):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        s.settimeout(0.3)
        return s.connect_ex((host, port)) == 0

# If something is already running on PORT, stop here (avoid two servers)
if is_port_open(HOST, PORT):
    raise RuntimeError(f"Port {PORT} is already in use on {HOST}. Stop the existing server or change PORT.")

server_cmd = [
    str(SERVER_BIN),
    "-m", str(MODEL_GGUF),
    "--host", HOST,
    "--port", str(PORT),
    "-c", str(CTX),
    "-ngl", str(NGL),
]

print("Starting llama-server:\n", " ".join(server_cmd))
server_proc = subprocess.Popen(server_cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

# Wait until server is ready
t0 = time.time()
ready = False
while time.time() - t0 < 180:
    try:
        r = requests.get(f"{BASE_URL}/models", timeout=2)
        if r.status_code == 200:
            ready = True
            break
    except Exception:
        pass
    time.sleep(1)

if not ready:
    print("Server did not become ready. Recent logs:")
    for _ in range(120):
        line = server_proc.stdout.readline()
        if not line:
            break
        print(line, end="")
    raise RuntimeError("llama-server failed to start")

print("✅ Server is ready:", BASE_URL)

# Print some startup logs (usually includes CUDA/offload info)
print("\n--- startup logs (first ~80 lines) ---")
for _ in range(80):
    line = server_proc.stdout.readline()
    if not line:
        break
    print(line, end="")
print("\n--- end logs ---")


Starting llama-server:
 /opt/llama.cpp/build/bin/llama-server -m /workspace/MentorApp/outputs/GGUF/qwen2_5_coder_7b_merged_f16.gguf --host 127.0.0.1 --port 8081 -c 4096 -ngl 99
✅ Server is ready: http://127.0.0.1:8081/v1

--- startup logs (first ~80 lines) ---
ggml_cuda_init: found 1 CUDA devices:
  Device 0: NVIDIA GeForce RTX 5090, compute capability 12.0, VMM: yes
main: n_parallel is set to auto, using n_parallel = 4 and kv_unified = true
build: 7987 (6948adc90) with GNU 13.3.0 for Linux x86_64
system info: n_threads = 32, n_threads_batch = 32, total_threads = 64

system_info: n_threads = 32 (n_threads_batch = 32) / 64 | CUDA : ARCHS = 1200 | USE_GRAPHS = 1 | PEER_MAX_BATCH_SIZE = 128 | BLACKWELL_NATIVE_FP4 = 1 | CPU : SSE3 = 1 | SSSE3 = 1 | AVX = 1 | AVX2 = 1 | F16C = 1 | FMA = 1 | BMI2 = 1 | LLAMAFILE = 1 | OPENMP = 1 | REPACK = 1 | 

Running without SSL
init: using 63 threads for HTTP server
start: binding port with default address family
main: loading model
srv    load_model: lo

In [105]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOpenAI(
    model="local-model",
    base_url=BASE_URL,     # llama-server OpenAI-compatible endpoint
    api_key="not-needed",
    temperature=0.0,
    max_tokens=MAX_TOKENS,
    # If your version supports it, it helps stop after a code fence closes:
)

print("✅ LangChain client ready.")


✅ LangChain client ready.


In [106]:
def server_chat(system: str, user: str, max_tokens: int = MAX_TOKENS, temperature: float = TEMP) -> str:
    """
    LangChain wrapper around llama-server.
    Returns the assistant text.
    """
    # Override per-call settings if needed
    local_llm = llm.bind(
        temperature=float(temperature),
        max_tokens=int(max_tokens),
    )

    msgs = [
        SystemMessage(content=system),
        HumanMessage(content=user),
    ]

    resp = local_llm.invoke(msgs)

    # LangChain returns an AIMessage-like object
    return (resp.content or "").strip()

### Prompt templates (Setting A)

This cell defines the system instruction and the user prompt format used for **Phase 1** inference.
In Phase 1, I only run **Setting A** (no traceback hint) and save the model’s predicted corrected code.


In [ ]:
SYSTEM_PROMPT = (
    "You are a Python bug fixer.\n"
    "Return ONLY this XML-like format (no extra text):\n"
    "<correct_code>\n...full corrected python code...\n</correct_code>\n"
    "<error_type>\n...one short line describing the original bug type...\n</error_type>\n"
)


def build_user_A(incorrect: str) -> str:
    return (
        "Incorrect code:\n"
        "```python\n"
        f"{incorrect}\n"
        "```"
    )


### Configuration and helper utilities

This cell collects the main **Phase 1** settings (sample count, generation limits, preview length) and defines helper functions used throughout the notebook:
- dataset normalization (`to_list_of_dicts`) to ensure I always iterate over a clean `list[dict]`,
- stable sample identifiers (`safe_sample_id`) for consistent folder naming,
- output parsing helpers to extract `<correct_code>` and `<error_type>` from the model response,
- lightweight preview and similarity utilities for quick sanity checks.


In [ ]:
from collections.abc import Mapping
from difflib import SequenceMatcher
import re

# -------------------------
# Controls (Phase 1: generate only)
# -------------------------
N_SAMPLES = len(eval_examples)            
PRINT_PREVIEW = True
PREVIEW_CHARS = 800
MAX_TOKENS = 1500
TEMP = 0.0

# -------------------------
# Helpers
# -------------------------
def similarity_ratio(a: str, b: str) -> float:
    a = a or ""
    b = b or ""
    return SequenceMatcher(None, a, b).ratio()

def safe_sample_id(i: int, ex: dict) -> str:
    title = ex.get("title") or f"sample_{i}"
    if not isinstance(title, str):
        title = str(title)
    title = title.strip().replace(" ", "_")
    title = re.sub(r"[^a-zA-Z0-9_\-]+", "", title)[:40]
    return f"{i:03d}_{title or 'sample'}"

def preview(s: str, n: int = PREVIEW_CHARS) -> str:
    s = s or ""
    return s if len(s) <= n else s[:n] + "\n... [TRUNCATED] ..."

def extract_tag(text: str, tag: str) -> str:
    text = text or ""
    m = re.search(rf"<{tag}>\s*(.*?)\s*</{tag}>", text, flags=re.DOTALL | re.IGNORECASE)
    return (m.group(1).strip() if m else "")

def extract_correct_code_and_error(raw: str):
    code = extract_tag(raw, "correct_code")
    err  = extract_tag(raw, "error_type")
    return code, err

ANSI_RE = re.compile(r"\x1b\[[0-9;]*m")
def strip_ansi(s: str) -> str:
    return ANSI_RE.sub("", s or "")

def strip_code_fences(s: str) -> str:
    s = (s or "").strip()
    s = re.sub(r"^\s*```[a-zA-Z0-9_-]*\s*", "", s)
    s = re.sub(r"\s*```\s*$", "", s)
    return s.strip()

def to_list_of_dicts(ds):
    # Case 1: already list[dict]
    if isinstance(ds, list):
        if len(ds) == 0:
            return []
        if isinstance(ds[0], dict):
            return ds
        raise TypeError("Got a list, but elements are not dicts.")

    # Case 2: dict-of-lists (common with HF Dataset slicing ds[:N])
    if isinstance(ds, Mapping):
        keys = list(ds.keys())
        if not keys:
            return []
        n = len(ds[keys[0]])
        return [{k: ds[k][i] for k in keys} for i in range(n)]

    # Case 3: HF Dataset-like object: use indexing
    try:
        _ = ds[0]
        return [ds[i] for i in range(len(ds))]
    except Exception as e:
        raise TypeError(f"Unsupported dataset type: {type(ds)}") from e

# ✅ Make a safe list-of-dicts for looping
eval_examples = to_list_of_dicts(eval_dataset) 

print(type(eval_examples), len(eval_examples))
print(type(eval_examples[0]), eval_examples[0].keys())


<class 'list'> 88
<class 'dict'> dict_keys(['title', 'description', 'difficulty', 'correct_code', 'incorrect_code', 'error_type'])


### Phase 1 generation loop: run inference and save per-sample artifacts

This cell iterates over the evaluation samples and performs **generation-only inference** (Setting A).
For each sample, it:
- sends the `incorrect_code` to the model through `llama-server`,
- parses the response to extract `<correct_code>` and `<error_type>`,
- saves the prediction and raw outputs into a dedicated folder under `eval_outputs/setting_A/<sid>/`,
- logs a compact record (latency, lengths, optional similarity) into a global `summary_phase1_generation.json`.

Errors during inference are caught and recorded per-sample so the loop can continue without interruption.


In [ ]:
# -------------------------
# Output dirs
# -------------------------
ensure_dir(OUT_DIR)
A_DIR = ensure_dir(OUT_DIR / "setting_A")

summary = []

# -------------------------
# Main loop (generate only)
# -------------------------
for i, ex in enumerate(eval_examples[:N_SAMPLES]):
    sid = safe_sample_id(i, ex)
    outA = ensure_dir(A_DIR / sid)   # create early so we can log exceptions

    try:
        incorrect = ex.get("incorrect_code", "")
        correct   = ex.get("correct_code", "")
        err_type  = ex.get("error_type", "")

        print("\n" + "="*90)
        print(f"[{i}] sid={sid}")

        promptA = build_user_A(incorrect)

        t0 = time.time()
        rawA = server_chat(system=SYSTEM_PROMPT, user=promptA, max_tokens=MAX_TOKENS, temperature=TEMP)
        latency = time.time() - t0

        if not (rawA or "").strip():
            predA = ""
            pred_err = ""
            status = "EMPTY_MODEL_OUTPUT"
        else:
            predA, pred_err = extract_correct_code_and_error(rawA)
            status = "OK" if predA.strip() else "UNPARSEABLE_OUTPUT"

        # Save artifacts (generation only)
        (outA / "prompt.txt").write_text(promptA, encoding="utf-8")
        (outA / "raw_model_output.txt").write_text(rawA or "", encoding="utf-8")
        (outA / "prediction.py").write_text(predA or "", encoding="utf-8")
        (outA / "predicted_error_type.txt").write_text(pred_err or "", encoding="utf-8")

        # Optional convenience files
        (outA / "incorrect.py").write_text(incorrect or "", encoding="utf-8")
        if correct:
            (outA / "correct.py").write_text(correct, encoding="utf-8")

        if PRINT_PREVIEW:
            print(f"Status: {status} | latency_s={latency:.2f} | pred_len={len(predA)}")
            if pred_err:
                print("Predicted error_type:", pred_err)
            print("Prediction preview:\n", preview(predA))

        row = {
            "sid": sid,
            "error_type": err_type,
            "status": status,
            "latency_s": round(latency, 3),

            "A_len": len(predA),
            "incorrect_len": len(incorrect),

            # Optional: similarity (cheap, but not a correctness proof)
            "A_sim_to_ref": (similarity_ratio(predA, correct) if correct and predA.strip() else None),

            # Model-predicted error type from tags
            "predicted_error_type": pred_err or None,
        }
        summary.append(row)

    except Exception as e:
        (outA / "GEN_EXCEPTION.txt").write_text(repr(e), encoding="utf-8")
        print(f" Generation exception for {sid}: {e}")

        summary.append({
            "sid": sid,
            "error_type": ex.get("error_type", ""),
            "status": "GEN_EXCEPTION",
            "latency_s": None,
            "A_len": None,
            "incorrect_len": len(ex.get("incorrect_code", "") or ""),
            "A_sim_to_ref": None,
            "predicted_error_type": None,
            "gen_exception": repr(e),
        })
        continue

# Save summary (generation only)
summary_path = OUT_DIR / "summary_phase1_generation.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print("\n Phase 1 done. Saved outputs to:", OUT_DIR)
print("Saved summary:", summary_path)



[0] sid=000_Titanic_Missing_Age_Imputation_using_Sim
Status: OK | latency_s=14.10 | pred_len=5493
Predicted error_type: LogicError
Prediction preview:
 import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

# Set random seed for reproducibility
np.random.seed(42)

# Load the Titanic dataset from seaborn
titanic = sns.load_dataset('titanic')

# Display basic information about the dataset
print("Titanic Dataset Shape:", titanic.shape)
print("\nFirst few rows:")
print(titanic.head())
print("\nMissing values per column:")
print(titanic.isnull().sum())

# Select relevant features for modeling
# We'll use: pclass, sex, age, sibsp, parch, fare
# Target: survived
features = ['pclass', 'sex', 'age', 'sibsp
... [TRUNCATED] ...

[1] sid=001_CIFAR-10_Deep

### Phase 2

Give the model the samples that could not correct them but this time with their error and trace backs. 

In [ ]:
SYSTEM_PROMPT_B = (
    "You are a Python bug fixer.\n"
    "You will be given a traceback tail and buggy Python code.\n"
    "Return ONLY this exact format and nothing else:\n"
    "<correct_code>\n"
    "(full corrected python code)\n"
    "</correct_code>\n"
    "<error_type>\n"
    "(one short line for the original bug type)\n"
    "</error_type>\n"
    "Do NOT use markdown. Do NOT include backticks. Do NOT echo the prompt."
)


def build_user_B(incorrect: str, tb_tail: str) -> str:
    return (
        "TRACEBACK_TAIL:\n"
        f"{tb_tail}\n"
        "\n"
        "BUGGY_CODE:\n"
        f"{incorrect}\n"
    )


In [ ]:
# Where Setting A outputs were saved (contains incorrect.py, prediction.py, etc.)
A_DIR = Path(r"C:\Users\hbahmanyar\MentorApp\Fine-Tuning\Qwen\Fine-Tune_Results\fine_tuned_eval_outputs")

# The runtime report you already produced for Setting A
SMOKE_REPORT_PATH = Path(r"C:\Users\hbahmanyar\MentorApp\Fine-Tuning\Qwen\Fine-Tune_Results\fine_tuned_eval_runtime_A\..")

# Output dir for Setting B generations
OUT_B = Path(r"C:\Users\hbahmanyar\MentorApp\Fine-Tuning\Qwen\Fine-Tune_Results\fine_tuned_outputs_B")
OUT_B.mkdir(parents=True, exist_ok=True)

smoke = json.loads(SMOKE_REPORT_PATH.read_text(encoding="utf-8"))

to_retry = [
    r for r in smoke
    if (r.get("ok") is False)  # includes failed and skipped
]

print("To retry in B (failed + skipped):", len(to_retry))


In [ ]:
def sid_from_smoke_row(row: dict) -> str:
    # Example row["file"]:
    # ...\Pre_Test_A_outputs\017_Reuters_News_Topic_Classification\prediction.py
    p = Path(row["file"])
    return p.parent.name  # folder name is sid

def load_pred_from_A(sid: str) -> str:
    p2 = A_DIR / sid / "prediction.py"
    return p2.read_text(encoding="utf-8", errors="ignore") if p2.exists() else ""


In [ ]:
# -------------------------
# Phase B: re-generate only failed samples using traceback tail
# -------------------------
to_process = [r for r in smoke if (r.get("ok") is False)]

for j, row in enumerate(to_process):
    sid = sid_from_smoke_row(row)
    outB = ensure_dir(B_DIR / sid)

    try:
        tb_tail = (row.get("stderr_tail", "") or "").strip()

        # If syntax check failed and you marked as skipped, inject a synthetic hint
        if row.get("skipped") is True and not tb_tail:
            tb_tail = "SyntaxError: the previous generated code failed to parse. Fix the syntax and produce runnable code."

        tb_tail_clean = strip_ansi(tb_tail)

        incorrect_B = load_pred_from_A(sid)  # the A prediction is now the "incorrect" for B
        promptB = build_user_B(incorrect_B, tb_tail_clean)

        t0 = time.time()
        rawB = server_chat(system=SYSTEM_PROMPT_B, user=promptB, max_tokens=MAX_TOKENS, temperature=TEMP)
        latency = time.time() - t0

        if not (rawB or "").strip():
            predB = ""
            pred_err = ""
            status = "EMPTY_MODEL_OUTPUT"
        else:
            predB, pred_err = extract_correct_code_and_error(rawB)
            status = "OK" if (predB or "").strip() else "UNPARSEABLE_OUTPUT"

        # Save artifacts
        (outB / "prompt.txt").write_text(promptB, encoding="utf-8")
        (outB / "raw_model_output.txt").write_text(rawB or "", encoding="utf-8")
        (outB / "prediction.py").write_text(predB or "", encoding="utf-8")
        (outB / "predicted_error_type.txt").write_text(pred_err or "", encoding="utf-8")

        # Carry runtime context forward
        (outB / "stderr_tail.txt").write_text(tb_tail or "", encoding="utf-8")
        (outB / "stdout_tail.txt").write_text((row.get("stdout_tail","") or ""), encoding="utf-8")

        if PRINT_PREVIEW:
            print("\n" + "="*90)
            print(f"[B {j}] sid={sid} | status={status} | latency_s={latency:.2f} | pred_len={len(predB or '')}")
            if pred_err:
                print("Predicted error_type:", pred_err)
            print("Prediction preview:\n", preview(predB))

        summary_B.append({
            "sid": sid,
            "status": status,
            "latency_s": round(float(latency), 3),
            "pred_len": len(predB or ""),
            "source_idx": row.get("idx"),
            "source_file": row.get("file"),
            "source_patched": row.get("patched"),
        })

    except Exception as e:
        (outB / "GEN_EXCEPTION.txt").write_text(repr(e), encoding="utf-8")
        print(f"[B] Generation exception for {sid}: {e}")

        summary_B.append({
            "sid": sid,
            "status": "GEN_EXCEPTION",
            "latency_s": None,
            "pred_len": None,
            "gen_exception": repr(e),
            "source_idx": row.get("idx"),
            "source_file": row.get("file"),
        })

# Save summary of phase B generation
summary_path_B = B_DIR / "summary_phaseB_generation.json"
summary_path_B.write_text(json.dumps(summary_B, indent=2), encoding="utf-8")
print("Saved Phase B summary:", summary_path_B)